# Pydantic格式的使用
## 1、基本使用
举例1：

In [ ]:
import os
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv
from scripts.regsetup import description

#1.读取.env配置文件信息,相关的环境变量以.env文件中的优先
load_dotenv(verbose=True)
DEEPSEEK_API_KEY=os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL=os.getenv("DEEPSEEK_BASE_URL")
#2.模型初始化
model=ChatDeepSeek(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL,
    # 关键修改：关闭思考模式
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    },
)

In [ ]:

from pydantic import BaseModel, Field


class Person(BaseModel):
    """人物信息"""
    name:str=Field(description="姓名")
    age:int=Field(description="年龄")
    occupation:str=Field(description="职业")
#创建结构化输出的大语言模型
structured_model=model.with_structured_output(Person)
result=structured_model.invoke("张三是一个30岁的软件工程师")
print(result)
print(type(result))


In [ ]:
print(f"姓名：{result.name}")
print(f"年龄：{result.age}")
print(f"职业：{result.occupation}")

举例2：

In [ ]:
class MovieModel(BaseModel):
    title:str=Field(description="电影标题")
    year:int=Field(description="发行年份")
    director:str=Field(description="电影导演")
    rating:float=Field(description="电影评分")
structured_model=model.with_structured_output(MovieModel)
result=structured_model.invoke("给出电影盗梦空间的信息")
print(result)
print(type(result))

## 2、高级特性
### 2.1情况1：可选字段
举例：

In [ ]:

from pydantic import BaseModel, Field


class Person(BaseModel):
    """人物信息"""
    name:str=Field(description="姓名")
    age:int=Field(description="年龄")
    occupation:str=Field(description="职业")
#创建结构化输出的大语言模型
structured_model=model.with_structured_output(Person)
result=structured_model.invoke("张三是一个软件工程师")
print(result)
print(type(result))


作为对比

In [ ]:

from typing import Optional
from pydantic import BaseModel, Field


class Person(BaseModel):
    """人物信息"""
    name:str=Field(description="姓名")
    age:Optional[int]=Field(description="年龄")
    occupation:str=Field(description="职业")
#创建结构化输出的大语言模型
structured_model=model.with_structured_output(Person)
result=structured_model.invoke("张三是一个软件工程师")
print(result)
print(type(result))


### 2.2情况2：默认值
不同的模型供应商，对于此字段的支持是不同的。比如：closeai平台的gpt-5.4-mini模型就不支持
此字段，而openrouter平台的gpt-5.4-mini模型就支持此字段。
自己调用看看不同平台对于同一模型默认值的支持

使用closeai平台

使用openrouter平台

### 2.3情况3：枚举类型
举例：方式1：

In [ ]:
from enum import Enum

#定义枚举类型
class Priority(str,Enum):
    LOW="低"
    MEDIUM="中"
    HIGH="高"
class CustomerInfo(BaseModel):
    """客户信息"""
    name: str = Field(description="客户姓名")
    phone: str = Field(description="电话号码")
    email: Optional[str] = Field(description="邮箱")
    issue: str = Field(description="问题描述")
    urgency: Priority = Field(description="紧急程度")
structured_llm = model.with_structured_output(CustomerInfo)
conversation = """
客服: 您好，请问有什么可以帮助您？
客户: 我是王小明，电话 138-1234-5678，我的订单一直没发货，很着急！
客服: 好的，我帮您查一下
"""
result = structured_llm.invoke(f"从以下客服对话中提取客户信息：\n{conversation}")
print(result)
print("\n提取结果：")
print(f" 客户: {result.name}")
print(f" 电话: {result.phone}")
print(f" 邮箱: {result.email or '未提供'}")
print(f" 问题: {result.issue}")
print(f" 紧急程度: {result.urgency.value}")

方式2：

In [ ]:
from typing import Literal
from enum import Enum
class CustomerInfo(BaseModel):
    """客户信息"""
    name: str = Field(description="客户姓名")
    phone: str = Field(description="电话号码")
    email: Optional[str] = Field(description="邮箱")
    issue: str = Field(description="问题描述")
    urgency: Literal["低","中","高"] = Field(description="紧急程度")
structured_llm = model.with_structured_output(CustomerInfo)
conversation = """
客服: 您好，请问有什么可以帮助您？
客户: 我是王小明，电话 138-1234-5678，我的订单一直没发货，很着急！
客服: 好的，我帮您查一下
"""
result = structured_llm.invoke(f"从以下客服对话中提取客户信息：\n{conversation}")
print(result)
print("\n提取结果：")
print(f" 客户: {result.name}")
print(f" 电话: {result.phone}")
print(f" 邮箱: {result.email or '未提供'}")
print(f" 问题: {result.issue}")
print(f" 紧急程度: {result.urgency}")

### 2.4情况4：列表提取
举例1：

In [ ]:
from typing import List


class Person(BaseModel):
    """人物信息"""
    name:str=Field(description="姓名")
    age:int=Field(description="年龄")
class PersonList(BaseModel):
    """人物列表"""
    people:List[Person]
structured_model=model.with_structured_output(PersonList)
result=structured_model.invoke("张三 30岁,李四 40岁")
print(result)


举例2：

In [ ]:
class Review(BaseModel):
    """产品评论"""
    product: str
    rating: int = Field(description="评分 1-5")
    pros: List[str] = Field(description="优点列表")
    cons: List[str] = Field(description="缺点列表")
structured_llm = model.with_structured_output(Review)
review = structured_llm.invoke("""
iPhone 17 很棒！摄像头强大，手感好。但是价格贵，没有充电器。4分。
""")
print(review)

举例3：

In [ ]:
class Invoice(BaseModel):
    """发票信息"""
    invoice_number: str = Field(description="发票号")
    date: str = Field(description="日期")
    total_amount: float = Field(description="总金额")
    items: List[str] = Field(description="商品")
# 测试
structured_llm = model.with_structured_output(Invoice)
invoice_text = """
发票号: INV-2024-001
日期: 2024-01-15
总金额: 1299.00
商品: MacBook Pro, AppleCare+
"""
invoice = structured_llm.invoke(f"提取发票信息：{invoice_text}")
print(invoice)

### 2.5情况5：嵌套结构
举例1：

In [ ]:
class Adress(BaseModel):
    """地点描述"""
    city:str=Field(description="城市")
    district:str=Field(description="区域")
class Company(BaseModel):
    """公司信息"""
    name:str=Field(description="公司名称")
    address:Adress=Field(description="公司所在地")
structured_model=model.with_structured_output(Company)
result=structured_model.invoke("阿里巴巴在杭州的滨江区")
print(result)

举例2:

In [ ]:
from typing import List
from pydantic import BaseModel, Field


class Actor(BaseModel):
    """演员信息"""
    name: str = Field(description="演员姓名")
    role: str = Field(description="演员饰演的角色")
class Movie(BaseModel):
    """电影信息"""
    title: str = Field(description="电影标题")
    year: int = Field(description="上映年份，必须填写整数")
    director: str = Field(description="导演姓名，必须填写")
    cast: List[Actor] = Field(
        description="主要演员列表，每位演员必须包含姓名和饰演角色"
    )
    rating: float = Field(
        description="电影评分"
    )

structured_model = model.with_structured_output(Movie)
response = structured_model.invoke(
    """
    请介绍电影《盗梦空间》。
    """
)
print(f"电影名：{response.title}")
print(f"上映年份：{response.year}")
print(f"导演：{response.director}")
print(f"评分：{response.rating}")
print(f"演员列表：{response.cast}")

举例3：

In [ ]:
from pydantic import BaseModel
from typing import List
class Aspect(BaseModel):
    """评论维度"""
    name: str = Field(description="维度名称，如：质量、价格、服务")
    score: int = Field(description="评分，1-5")
    comment: str = Field(description="具体评价")
class Sentiment(str,Enum):
    positive="积极正向"
    negative="消极负向"
    neutral="中立"
class ProductReview(BaseModel):
     """产品评论分析"""
     overall_sentiment: Sentiment = Field(description="整体情感")
     overall_score: int = Field(description="综合评分，1-5")
     aspects: List[Aspect] = Field(description="各维度评价")
     summary: str = Field(description="一句话总结")

# 创建结构化模型
structured_model = model.with_structured_output(ProductReview)
# 测试
review_text = """
这款笔记本电脑性能非常强大，运行大型软件毫无压力。
屏幕色彩鲜艳，看视频很舒服。
不过价格有点贵，而且风扇噪音较大。
客服态度很好，物流也快。
总体来说还是值得购买的。
"""
result = structured_model.invoke(
f"分析以下产品评论：\n{review_text}"
)
print(f"整体情感: {result.overall_sentiment}")
print(f"综合评分: {result.overall_score}/5")
print(f"\n各维度评价:")
for aspect in result.aspects:
    print(f" - {aspect.name}: {aspect.score}/5 - {aspect.comment}")
print(f"\n总结: {result.summary}")

### 2.6情况6：限制条件

In [ ]:
from pydantic import ValidationError
class User(BaseModel):
    name:str=Field(description="姓名",min_length=20,max_length=50)
    age:int=Field(description="年龄",le=150)
    email:str=Field(description="邮箱")
try:
    user1=User(name="tom",age=20,email="tom@163.com")
    print(f"[OK]{user1}]")
except ValidationError as e:
    print(f"[FAIL]{e}")

In [ ]:
from pydantic import ValidationError

try:
    user2=User(name="tom",age=200,email="tom@163.com")
    print(f"[OK]{user2}]")
except ValidationError as e:
    print(f"[FAIL]{e}")

举例1：
使用closeai平台的模型
使用openrouter平台的模型
自己对比